In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("master_cyclistic_data.csv")
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,member_casual,ride_length_mins,day_of_week
0,A835B6FFB85FF2F8,electric_bike,2026-07-15 20:19:48.831,2026-07-15 21:22:00.281,Burnham Greenway & 105th St,CHI00863,Burnham Greenway & 105th St,CHI00863,casual,62.190833,Wednesday
1,880813FDABFBAEF9,classic_bike,2026-07-10 23:35:22.204,2026-07-10 23:44:25.580,California Ave & Milwaukee Ave,CHI00307,California Ave & Division St,CHI00340,casual,9.056267,Friday
2,12AB6CD18F6C311D,classic_bike,2026-07-25 00:22:16.313,2026-07-25 00:23:43.303,California Ave & Milwaukee Ave,CHI00307,California Ave & Milwaukee Ave,CHI00307,casual,1.449833,Saturday
3,B79A9800B7B4DDA9,classic_bike,2026-07-04 11:18:15.715,2026-07-04 11:22:56.951,Granville Ave & Clark St,CHI02156,Granville Ave & Clark St,CHI02156,casual,4.687267,Saturday
4,99AD35C0916D16A8,classic_bike,2026-07-15 18:36:05.398,2026-07-15 19:49:07.601,Burnham Greenway & 105th St,CHI00863,Burnham Greenway & 105th St,CHI00863,casual,73.036717,Wednesday


In [3]:
# adding month column - It will run thru' Oct'25 to Sep'26 - but stored only as Oct, Nov, Dec, Jan....Sep
df['started_at'] = pd.to_datetime(df['started_at'])
df['month'] = df['started_at'].dt.month_name()
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,member_casual,ride_length_mins,day_of_week,month
0,A835B6FFB85FF2F8,electric_bike,2026-07-15 20:19:48.831,2026-07-15 21:22:00.281,Burnham Greenway & 105th St,CHI00863,Burnham Greenway & 105th St,CHI00863,casual,62.190833,Wednesday,July
1,880813FDABFBAEF9,classic_bike,2026-07-10 23:35:22.204,2026-07-10 23:44:25.580,California Ave & Milwaukee Ave,CHI00307,California Ave & Division St,CHI00340,casual,9.056267,Friday,July
2,12AB6CD18F6C311D,classic_bike,2026-07-25 00:22:16.313,2026-07-25 00:23:43.303,California Ave & Milwaukee Ave,CHI00307,California Ave & Milwaukee Ave,CHI00307,casual,1.449833,Saturday,July
3,B79A9800B7B4DDA9,classic_bike,2026-07-04 11:18:15.715,2026-07-04 11:22:56.951,Granville Ave & Clark St,CHI02156,Granville Ave & Clark St,CHI02156,casual,4.687267,Saturday,July
4,99AD35C0916D16A8,classic_bike,2026-07-15 18:36:05.398,2026-07-15 19:49:07.601,Burnham Greenway & 105th St,CHI00863,Burnham Greenway & 105th St,CHI00863,casual,73.036717,Wednesday,July


In [11]:
# Top 10 starting stations overall

df[['start_station_name']].value_counts().head(10)

start_station_name                
Navy Pier                             58932
DuSable Lake Shore Dr & Monroe St     33174
Michigan Ave & Oak St                 29908
DuSable Lake Shore Dr & North Blvd    27525
State St & Chicago Ave                25378
Canal St & Madison St                 25363
Theater on the Lake                   23310
Wells St & Concord Ln                 23305
Millennium Park                       22430
Wells St & Elm St                     22345
Name: count, dtype: int64

In [24]:
# Bike type preference by rider type

bike_preference = df.groupby('member_casual')['rideable_type'].value_counts()
bike_preference = bike_preference.unstack()
bike_preference

rideable_type,classic_bike,electric_bike
member_casual,,
casual,544261,852213
member,1234524,1418447


In [25]:
bike_preference['total'] = bike_preference['classic_bike'] + bike_preference['electric_bike']
bike_preference['classic_percentage'] = (bike_preference['classic_bike'] / bike_preference['total'])* 100
bike_preference['electric_percentage'] = (bike_preference['electric_bike'] / bike_preference['total'])* 100

In [26]:
bike_preference

rideable_type,classic_bike,electric_bike,total,classic_percentage,electric_percentage
member_casual,,,,,
casual,544261,852213,1396474,38.973944,61.026056
member,1234524,1418447,2652971,46.533641,53.466359


In [4]:
def sort_days(day):
    if day in ['Saturday','Sunday']:
        return 'Weekend'
    else:
        return 'Weekday'

df['day_type'] = df['day_of_week'].apply(sort_days)

In [9]:
df.groupby(['day_type','member_casual'])['ride_id'].count().unstack()

member_casual,casual,member
day_type,,
Weekday,876110,2054315
Weekend,520364,598656


In [16]:
month_order = ['October', 'November', 'December', 'January',
               'February', 'March', 'April', 'May',
               'June', 'July', 'August', 'September']

ww_df = df.groupby(['day_type', 'member_casual', 'month'])['ride_id'].count().unstack().reindex(columns=month_order)

In [18]:
ww_df = ww_df.unstack('day_type')

In [21]:
weekday = ww_df.xs('Weekday', axis=1, level='day_type')
weekend = ww_df.xs('Weekend', axis=1, level='day_type')

weekend_pct = weekend / (weekday + weekend) * 100

In [22]:
weekend_pct

month,October,November,December,January,February,March,April,May,June,July,August,September
member_casual,,,,,,,,,,,,
casual,37.266845,40.187371,25.917382,25.495578,27.048929,40.080938,30.689133,43.306765,38.262578,35.593681,39.910745,34.972389
member,21.259252,24.139438,16.499765,17.106451,17.227326,23.570415,19.303611,27.533628,22.802403,21.985767,26.568339,21.150886


In [23]:
weekday_pct = weekday / (weekday + weekend) * 100
weekday_pct

month,October,November,December,January,February,March,April,May,June,July,August,September
member_casual,,,,,,,,,,,,
casual,62.733155,59.812629,74.082618,74.504422,72.951071,59.919062,69.310867,56.693235,61.737422,64.406319,60.089255,65.027611
member,78.740748,75.860562,83.500235,82.893549,82.772674,76.429585,80.696389,72.466372,77.197597,78.014233,73.431661,78.849114


In [26]:
df.groupby('member_casual')['ride_length_mins'].agg(['mean','median'])

,mean,median
member_casual,,
casual,20.302232,11.967208
member,12.402564,8.699133


In [30]:
df.head(2)

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,member_casual,ride_length_mins,day_of_week,month,day_type
0,A835B6FFB85FF2F8,electric_bike,2026-07-15 20:19:48.831,2026-07-15 21:22:00.281,Burnham Greenway & 105th St,CHI00863,Burnham Greenway & 105th St,CHI00863,casual,62.190833,Wednesday,July,Weekday
1,880813FDABFBAEF9,classic_bike,2026-07-10 23:35:22.204,2026-07-10 23:44:25.580,California Ave & Milwaukee Ave,CHI00307,California Ave & Division St,CHI00340,casual,9.056267,Friday,July,Weekday


In [29]:
df.groupby(['member_casual','day_type'])['ride_length_mins'].agg(['mean','median'])

mean     median
member_casual day_type                      
casual        Weekday   18.518322  10.914892
              Weekend   23.305710  14.054483
member        Weekday   12.007291   8.493050
              Weekend   13.758961   9.521633

Final Data Quality Checks

In [32]:
df['ride_id'].duplicated().sum()

np.int64(19)

In [37]:
# Check of the ride_id column identified 19 duplicate entries.

df[df['ride_id'].duplicated(keep=False)].sort_values('ride_id').head(2)

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,member_casual,ride_length_mins,day_of_week,month,day_type
2816613,10117825A92C182D,classic_bike,2026-04-30 23:56:24.321,2026-05-01 00:01:51.860,Peoria St & Kinzie St,CHI02098,Ogden Ave & Chicago Ave,CHI00275,casual,5.458983,Thursday,April,Weekday
1835039,10117825A92C182D,classic_bike,2026-04-30 23:56:24.321,2026-05-01 00:01:51.860,Peoria St & Kinzie St,CHI02098,Ogden Ave & Chicago Ave,CHI00275,casual,5.458983,Thursday,April,Weekday


In [38]:
df[df['ride_id'].duplicated(keep=False)].sort_values('ride_id').count()

ride_id               38
rideable_type         38
started_at            38
ended_at              38
start_station_name    38
start_station_id      38
end_station_name      38
end_station_id        38
member_casual         38
ride_length_mins      38
day_of_week           38
month                 38
day_type              38
dtype: int64

In [40]:
# Check for missing values
df.isna().sum()

ride_id               0
rideable_type         0
started_at            0
ended_at              0
start_station_name    0
start_station_id      0
end_station_name      0
end_station_id        0
member_casual         0
ride_length_mins      0
day_of_week           0
month                 0
day_type              0
dtype: int64

In [43]:
# Check for zero or negative ride length

df[df['ride_length_mins']<=0]

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,member_casual,ride_length_mins,day_of_week,month,day_type


In [44]:
# Check for unusually long rides

df[df['ride_length_mins']>1440]

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,member_casual,ride_length_mins,day_of_week,month,day_type
